# 602 address-only LSTM gate vs streamer

This notebook trains and performs causal offline inference for one trace only: `602.gcc_s-734B`. The matched normal policy is the pinned ChampSim streamer (64 page trackers, degree 5). Both policies use only the current cache-line address and causal prior address history. PC is retained only in the replay key and never enters the model. ChampSim is run later on Linux.

The five widths are a predeclared exploratory capacity sweep. Select a width only after Linux replay, then confirm it on a fresh held-out window.

In [ ]:
import os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU (A100 preferred)'
torch.set_float32_matmul_precision('high')
print(torch.cuda.get_device_name(0), torch.__version__)

REPO = '/content/cache_arch'
PUBLIC_URL = 'https://github.com/Angelawoo572/cache_arch.git'
TOKEN = userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add a private-repository GITHUB_TOKEN in Colab Secrets'
ASKPASS = '/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n')
os.chmod(ASKPASS, 0o700)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': ASKPASS, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': TOKEN})
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', PUBLIC_URL, REPO], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', 'main'], check=True, env=git_env)
finally:
    pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ID = '602_offline_lstm_streamer_sweep_seed7'
DRIVE_ROOT = f'/content/drive/MyDrive/cache_prefetch_602_streamer/{RUN_ID}'
INPUT_DIR = f'{DRIVE_ROOT}/colab_input'
OUTPUT_ROOT = f'{DRIVE_ROOT}/colab_output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
INPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_input.tar.gz'
assert os.path.isfile(INPUT_ARCHIVE), INPUT_ARCHIVE
with tarfile.open(INPUT_ARCHIVE, 'r:gz') as archive:
    archive.extractall(INPUT_DIR)
print('Input:', INPUT_ARCHIVE)
print('Persistent output:', OUTPUT_ROOT)


In [ ]:
import gzip, hashlib, json
TRACE = '602.gcc_s-734B'
TRAIN = f'{INPUT_DIR}/{TRACE}.train_stream.csv.gz'
EVAL = f'{INPUT_DIR}/{TRACE}.eval_stream.csv.gz'
for path in (TRAIN, EVAL):
    assert os.path.isfile(path), path
    with gzip.open(path, 'rb') as handle:
        digest = hashlib.sha256(handle.read()).hexdigest()
    print(path, digest)
SCRIPT = f'{REPO}/formal_NN_training/experiments/602_offline_lstm_streamer/python/train_and_offline_infer.py'
assert os.path.isfile(SCRIPT), SCRIPT


In [ ]:
HIDDEN_SIZES = [8, 16, 32, 64, 128]
SWEEP = []
for hidden_size in HIDDEN_SIZES:
    tag = f'h{hidden_size}'
    out_dir = f'{OUTPUT_ROOT}/{tag}'
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    cmd = [
        sys.executable, SCRIPT,
        '--train-stream', TRAIN,
        '--eval-stream', EVAL,
        '--out-dir', out_dir,
        '--device', 'cuda',
        '--seed', '7',
        '--epochs', '8',
        '--chunk-len', '1024',
        '--batch-chunks', '32',
        '--hidden-size', str(hidden_size),
    ]
    print('Training', tag)
    subprocess.run(cmd, check=True)
    metadata = json.loads(pathlib.Path(f'{out_dir}/run_metadata.json').read_text())
    assert metadata['matched_normal_prefetcher'] == 'streamer'
    assert metadata['model_does_not_use_pc'] is True
    SWEEP.append({
        'model_tag': tag,
        'hidden_size': hidden_size,
        'parameter_count': metadata['parameter_count'],
        'threshold': metadata['threshold'],
        'offline_streamer_entries': metadata['offline_streamer_entries'],
        'offline_lstm_entries': metadata['offline_lstm_entries'],
    })
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(
    json.dumps({'trace': TRACE, 'matched_normal_prefetcher': 'streamer', 'points': SWEEP}, indent=2) + '\n'
)
print(json.dumps(SWEEP, indent=2))


In [ ]:
required = ['offline_streamer.replay.csv', 'offline_lstm.replay.csv', 'model.pt', 'run_metadata.json']
for hidden_size in HIDDEN_SIZES:
    out_dir = f'{OUTPUT_ROOT}/h{hidden_size}'
    assert all(os.path.isfile(f'{out_dir}/{name}') for name in required), out_dir

OUTPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
LOCAL_STAGE = f'/content/{RUN_ID}_colab_output_stage'
LOCAL_ARCHIVE = f'/content/{RUN_ID}.colab_output.tar.gz'
if os.path.isdir(LOCAL_STAGE):
    shutil.rmtree(LOCAL_STAGE)
shutil.copytree(OUTPUT_ROOT, LOCAL_STAGE)
with tarfile.open(LOCAL_ARCHIVE, 'w:gz') as archive:
    archive.add(LOCAL_STAGE, arcname='.')
shutil.copy2(LOCAL_ARCHIVE, OUTPUT_ARCHIVE)
print('DONE:', OUTPUT_ARCHIVE, os.path.getsize(OUTPUT_ARCHIVE), 'bytes')
